In [ ]:
#Install Required Libraries

!pip install -q transformers datasets accelerate timm scikit-learn pillow

In [ ]:
#Import Required Libraries

import os
import json
import time
import torch
import random
import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from transformers import (
    VisualBertModel,
    VisualBertConfig,
    BertTokenizer,
    TrainingArguments,
    Trainer
)

import torch.nn as nn
from torch.utils.data import Dataset

In [ ]:
#Set Device and Seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

In [ ]:
#Download Hate Meme Dataset from KaggleHub

import kagglehub

# Download dataset

dataset_path = kagglehub.dataset_download(
    "parthplc/facebook-hateful-meme-dataset"
)

print("Dataset Path:", dataset_path)

In [ ]:
#Load Dataset Paths

TRAIN_FILE = os.path.join(
    dataset_path,
    "data/train.jsonl"
)

DEV_FILE = os.path.join(
    dataset_path,
    "data/dev.jsonl"
)

TEST_FILE = os.path.join(
    dataset_path,
    "data/test.jsonl"
)

IMAGE_DIR = os.path.join(
    dataset_path,
    "data"
)

print(TRAIN_FILE)
print(DEV_FILE)
print(TEST_FILE)

In [ ]:
#Load JSONL Files

def load_jsonl(file_path, limit=None):

    data = []

    with open(file_path, "r") as f:

        for idx, line in enumerate(f):

            if limit and idx >= limit:
                break

            sample = json.loads(line)

            data.append(sample)

    return data


train_data = load_jsonl(TRAIN_FILE)

dev_data = load_jsonl(DEV_FILE)

test_data = load_jsonl(TEST_FILE)

print("TRAIN:", len(train_data))
print("DEV:", len(dev_data))
print("TEST:", len(test_data))

In [ ]:
train_data = train_data[:1000]

dev_data = dev_data[:500]

test_data = test_data[:500]

print("TRAIN:", len(train_data))
print("DEV:", len(dev_data))
print("TEST:", len(test_data))

In [ ]:
#Initialize Tokenizer

MODEL_NAME = "uclanlp/visualbert-vqa-coco-pre"

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
#Create Dataset Class

class HateMemeDataset(Dataset):

    def __init__(self, data, tokenizer, image_dir, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.image_dir = image_dir
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def extract_visual_features(self, image):

        image = image.resize((224, 224))

        image_array = np.array(image).astype(np.float32)

        if len(image_array.shape) == 2:
            image_array = np.stack([image_array] * 3, axis=-1)

        image_array = image_array / 255.0

        visual_embeds = torch.tensor(image_array).permute(2, 0, 1)

        visual_embeds = visual_embeds.mean(dim=0)

        visual_embeds = visual_embeds.flatten()

        if visual_embeds.shape[0] < 2048:
            pad_size = 2048 - visual_embeds.shape[0]
            visual_embeds = torch.cat([
                visual_embeds,
                torch.zeros(pad_size)
            ])

        visual_embeds = visual_embeds[:2048]

        visual_embeds = visual_embeds.unsqueeze(0)

        return visual_embeds

    def __getitem__(self, idx):

        sample = self.data[idx]

        text = sample["text"]
        label = sample["label"]

        image_path = os.path.join(
            self.image_dir,
            sample["img"]
        )

        image = Image.open(image_path).convert("RGB")

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        visual_embeds = self.extract_visual_features(image)

        visual_attention_mask = torch.ones(
            visual_embeds.shape[:-1],
            dtype=torch.long
        )

        visual_token_type_ids = torch.ones(
            visual_embeds.shape[:-1],
            dtype=torch.long
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "token_type_ids": encoding["token_type_ids"].squeeze(0),
            "visual_embeds": visual_embeds,
            "visual_attention_mask": visual_attention_mask,
            "visual_token_type_ids": visual_token_type_ids,
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [ ]:
#Create Train / Dev / Test Datasets

train_dataset = HateMemeDataset(
    train_data,
    tokenizer,
    IMAGE_DIR
)

val_dataset = HateMemeDataset(
    dev_data,
    tokenizer,
    IMAGE_DIR
)

test_dataset = HateMemeDataset(
    test_data,
    tokenizer,
    IMAGE_DIR
)

In [ ]:
#Create VisualBERT Classification Model

class VisualBERTClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.visualbert = VisualBertModel.from_pretrained(
            MODEL_NAME
        )

        hidden_size = self.visualbert.config.hidden_size

        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(hidden_size, 2)

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids,
        visual_embeds,
        visual_attention_mask,
        visual_token_type_ids,
        labels=None
    ):

        outputs = self.visualbert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            visual_embeds=visual_embeds,
            visual_attention_mask=visual_attention_mask,
            visual_token_type_ids=visual_token_type_ids
        )

        pooled_output = outputs.last_hidden_state[:, 0]

        pooled_output = self.dropout(pooled_output)

        logits = self.classifier(pooled_output)

        loss = None

        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits
        }

In [ ]:
#Initialize Model

model = VisualBERTClassifier().to(DEVICE)

In [ ]:
#Define Metrics Function

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    probs = torch.softmax(
        torch.tensor(logits),
        dim=1
    )[:, 1].numpy()

    accuracy = accuracy_score(labels, predictions)

    precision = precision_score(labels, predictions)

    recall = recall_score(labels, predictions)

    f1 = f1_score(labels, predictions)

    roc_auc = roc_auc_score(labels, probs)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc
    }

In [ ]:
#Configure Training Arguments

training_args = TrainingArguments(
    output_dir="./visualbert_results",

    num_train_epochs=10,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,

    weight_decay=0.01,

    logging_dir="./logs",

    logging_steps=50,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

In [ ]:
#Create Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
# Train VisualBERT

import time

start_train_time = time.time()

trainer.train()

end_train_time = time.time()

training_time = (
    end_train_time -
    start_train_time
)

print(
    f"Training Time: {training_time:.2f} seconds"
)

In [ ]:
#Save Trained Model

#trainer.save_model("visualbert_hatememe_model")

#tokenizer.save_pretrained("visualbert_hatememe_model")

#print("Model saved successfully")

In [ ]:
#Evaluate on Test Dataset

predictions = trainer.predict(val_dataset)

logits = predictions.predictions
labels = predictions.label_ids

preds = np.argmax(logits, axis=1)

probs = torch.softmax(
    torch.tensor(logits),
    dim=1
)[:, 1].numpy()

In [ ]:
#Calculate Performance Metrics

accuracy = accuracy_score(labels, preds)

precision = precision_score(labels, preds)

recall = recall_score(labels, preds)

f1 = f1_score(labels, preds)

roc_auc = roc_auc_score(labels, probs)

cm = confusion_matrix(labels, preds)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC-AUC:", roc_auc)

print("\nConfusion Matrix:\n")
print(cm)

In [ ]:
#Detailed Classification Report

print(
    classification_report(
        labels,
        preds,
        digits=4
    )
)

In [ ]:
#Measure Inference Time

start_inference = time.time()

predictions= trainer.predict(val_dataset)

end_inference = time.time()

inference_time = end_inference - start_inference

avg_inference = inference_time / len(val_dataset)

print(f"Total Inference Time: {inference_time:.4f} seconds")
print(f"Average Time Per Sample: {avg_inference:.6f} seconds")

In [ ]:
#Save Results to CSV

results_df = pd.DataFrame([
    {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc,
        "Inference Time": inference_time,
        "Avg Inference Time": avg_inference
    }
])

results_df.to_csv(
    "visualbert_test_results.csv",
    index=False
)

print(results_df)

In [ ]:
#Predict on Single Sample After Training

def predict_single(text, image_path):

    model.eval()

    image = Image.open(image_path).convert("RGB")

    imag